# Lesson 2 — Tokens & embeddings (runnable)

Computers don't understand "dog". They understand numbers. We do two things:
**tokenise** (each word → integer id), then **embed** (each id → vector of numbers).

This notebook is a runnable version of [`02_tokens_and_embeddings.py`](../02_tokens_and_embeddings.py).
For the intuition (kid checklist analogy, dot products), see [`02_walkthrough.md`](../02_walkthrough.md).


## Imports

In [ ]:
import torch                                # PyTorch tensors.
import torch.nn as nn                       # nn.Embedding lives here.

torch.manual_seed(0)                        # Reproducible RNG.
torch.set_printoptions(precision=2, sci_mode=False)

## Step 1 — Tokenise

A vocabulary list. Each word's position is its ID.

In [ ]:
vocab = ["<pad>", "<mask>", "dog", "cat", "fish", "bark", "meow", "swim"]
tok2id = {w: i for i, w in enumerate(vocab)}     # word → id lookup.

print("Vocabulary:")
for i, w in enumerate(vocab):
    print(f"  id={i}  word={w}")

# Encoding a sentence = look up each word's ID
sentence = ["dog", "bark"]
ids = [tok2id[w] for w in sentence]              # Convert each word to id.
print(f"\nSentence {sentence}  ->  ids {ids}")

## Step 2 — Embed

The embedding table is just a 2-D matrix. Row `i` = vector for word `i`.

In [ ]:
embedding_dim = 4                              # 4 numbers per word.
emb = nn.Embedding(len(vocab), embedding_dim)  # 8×4 trainable lookup table.

print(f"Embedding table shape: {tuple(emb.weight.shape)}")
print(f"  ({len(vocab)} words × {embedding_dim} numbers each)\n")

print("Word -> vector:")
for word in ["dog", "cat", "fish"]:
    v = emb(torch.tensor(tok2id[word])).detach()
    print(f"  {word:5s} ->  {v}")

**These vectors are RANDOM right now.** Embeddings only become meaningful during training. The model nudges them so words used in similar contexts end up near each other.

## Step 3 — Encoding a whole sentence at once

In [ ]:
ids_tensor = torch.tensor(ids)
vectors = emb(ids_tensor)                       # Shape (2, 4).
print(f"Sentence vectors:  shape {tuple(vectors.shape)}")
print(vectors.detach())

## Step 4 — Similarity via dot product

Multiply matching slots, add the products. Big total = similar; small or negative = different.

(Random embeddings → meaningless numbers. After training they'd reflect real word relationships.)

In [ ]:
def similarity(a, b):
    return torch.dot(emb(torch.tensor(tok2id[a])), emb(torch.tensor(tok2id[b]))).item()

print("Similarity (random embeddings):")
print(f"  dog · cat   = {similarity('dog', 'cat'):.2f}")
print(f"  dog · fish  = {similarity('dog', 'fish'):.2f}")
print(f"  bark · meow = {similarity('bark', 'meow'):.2f}")

## Things to try

1. Change `embedding_dim` to 16. What's the new shape of the embedding table?
2. Print the vector for `"<mask>"`. It's random too — but after training it would end up at a "neutral" point.
3. Add three new words: `"bird"`, `"tweet"`, `"fly"`. Encode `["bird", "tweet"]`.
4. (Stretch) Write a loop that finds, for each word, the OTHER word in the vocab with the highest dot product.